In [ ]:
import torch
import  torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import tiktoken
import requests
from math import sqrt

In [ ]:
class TextDataset(Dataset):
    
    def __init__(self, raw_text_corpus:str, context_length:int):
        self.tokenizer = tiktoken.get_encoding("gpt2")
        self.token_ids = self.tokenizer.encode(raw_text_corpus)
        self.context_length = context_length
        
    def __len__(self):
        return len(self.token_ids) - self.context_length
    
    def __getitem__(self, idx):
        chunk = self.token_ids[idx:idx+self.context_length+1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        
        # Multi-Head Attention
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        
        # Feedforward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        # Layer Normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        
        # Multi-Head Attention with Residual Connection
        residual = x
        x = self.norm1(x)
        x, _ = self.attention(x, x, x, attn_mask=mask)
        x = self.dropout(x) + residual
        
        # Feedforward Network with Residual Connection
        residual = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.dropout(x) + residual
        
        #output
        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size=50257, d_model=512, n_heads=8, d_ff=2048, n_layers=6, dropout=0.1, max_len=512):
        super().__init__()
        
        # Embeddings
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.tok_emb.weight, mean=0, std=1/sqrt(d_model))
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.scale = sqrt(d_model)
        self.drop = nn.Dropout(dropout)
        
        # Stack of 6 identical transformer blocks
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range (n_layers)])
        
        # Final output
        self.final_form = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.head.weight = self.tok_emb.weight 
   
        
    def forward(self, x):
        B, T = x.size()
        
        tok =  self.tok_emb(x) * self.scale
        pos = self.pos_emb(torch.arange(T, device=x.device))
        x = self.drop(tok + pos)
        
        #Casual mask to ensure that attention is only applied to previous tokens in the sequence
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        
        # Pass through the stack of transformer blocks
        for block in self.blocks:
            x = block(x, mask=mask)
            
        x  = self.final_form(x)
        logits = self.head(x)
        
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = Transformer(vocab_size=50257).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
raw_text_corpus = requests.get(url).text
context_length = 64
dataset = TextDataset(raw_text_corpus, context_length)

In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    pin_memory=True,
    drop_last=True
)

In [ ]:
#  Training loop
model.train()
epochs = 1
for epoch in range(epochs):
    for batch_idx, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        
        #forward pass
        logits = model(x)
        
        #loss
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
        
        #backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if  batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx}/{len(dataloader)}], Loss: {loss.item():.4f}")

In [ ]:
#  Text Generation
def generate_text(model, starting_text, max_new_tokens=50, context_length=64, temperature=1.0):
    
    model.eval()
    tokenizer = tiktoken.get_encoding("gpt2")
    if isinstance(starting_text, str):
        starting_text = [starting_text]
    input_ids = tokenizer.encode(starting_text[0])
    
    x = torch.tensor(input_ids, device=device, dtype=torch.long).unsqueeze(0)
    
    print("Generating text...\n")
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            
            y = x[:, -context_length:]  # Keep only the last context_length tokens
            logits = model(y)
            logits = logits[:, -1, :]  # Get the logits for the last token
            logits = logits / temperature  # Apply temperature
            probabilities = F.softmax(logits, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1)
            x = torch.cat((x, next_token_id), dim=1)
            
            # Print the generated token
            generated_token = tokenizer.decode(next_token_id.squeeze().tolist())
            print(generated_token, end='', flush=True)
            
    print("Done!")

In [ ]:
generate_text(model, starting_text=" O Romeo, O Romeo! wherefore", max_new_tokens=100, context_length=context_length)